<a href="https://colab.research.google.com/github/e23323-dot/Statistical-Learning-e23323/blob/main/Assignmen_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'

print("Q1. Bayesian Estimation of a User Ability Parameter from Item Responses")
print("="*70)

print("""
1. Visualizing the Mechanics:

P(Y_i = 1 | Θ = θ) = p_i(θ) = 1 / (1 + e^{-a_i(θ - b_i)})

where:
- a_i > 0 is the discrimination parameter
- b_i is the difficulty parameter

Interpretation:
- Increasing b_i shifts the curve right (harder item)
- Increasing a_i makes the curve steeper (more discriminating)
""")

theta_range = np.linspace(-4, 4, 500)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Different b values (a=1)", "Different a values (b=0)"))

for b in [-1, 0, 1]:
    p = 1/(1 + np.exp(-1*(theta_range - b)))
    fig.add_trace(go.Scatter(x=theta_range, y=p, mode='lines', name=f'b={b}'), row=1, col=1)

for a in [0.5, 1, 2]:
    p = 1/(1 + np.exp(-a*(theta_range - 0)))
    fig.add_trace(go.Scatter(x=theta_range, y=p, mode='lines', name=f'a={a}'), row=1, col=2)

fig.update_layout(height=400, width=800, title_text="Item Response Functions")
fig.show()

print("""
2. Sequential Likelihood Contribution:

For a single response y_k:
L(y_k | θ) = [p_k(θ)]^{y_k} [1 - p_k(θ)]^{1-y_k}

where y_k ∈ {0, 1}

For the running history vector y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = ∏_{i=1}^k [p_i(θ)]^{y_i} [1 - p_i(θ)]^{1-y_i}

3. Mathematical Formulation of the Running Update:

f_{Θ | Y^(k)}(θ | y^(k)) ∝ L(y_k | θ) · f_{Θ | Y^(k-1)}(θ | y^(k-1))

This recursive relationship uses the posterior from step (k-1) as the prior for step k.

4. Dynamic Shifting:

When y_k = 1 (correct answer) to a difficult item (large b_k):
- p_k(θ) is small for low θ and large for high θ
- Multiplying the prior by p_k(θ) shifts probability mass rightward
- The posterior peak moves toward higher θ values

When y_k = 0 (incorrect answer):
- [1 - p_k(θ)] is large for low θ and small for high θ
- The posterior peak shifts leftward

5. Tracking Certainty and Sharpness:

The discrimination parameter a_k controls the likelihood function shape:

Case 1: Very large a_k (a_k → ∞)
- p_k(θ) approaches a step function
- Likelihood becomes highly concentrated around θ = b_k
- Posterior variance decreases significantly (high certainty)

Case 2: Very small a_k (a_k → 0)
- p_k(θ) approaches 0.5 for all θ
- Likelihood provides almost no information
- Posterior variance remains large (low certainty)

6. Numerical Implementation of Running Grid:

Algorithm:
1. Define a fixed grid of θ values: {θ_1, θ_2, ..., θ_M}
2. Initialize prior: f_0(θ_j) = (1/√(2π)) exp(-θ_j²/2)
3. Normalize: f_0(θ_j) = f_0(θ_j) / Σ_m f_0(θ_m)Δθ

For each observation k = 1, 2, ..., n:
   a. Compute likelihood at each grid point:
      L(y_k|θ_j) = [p_k(θ_j)]^{y_k} [1 - p_k(θ_j)]^{1-y_k}

   b. Compute unnormalized posterior:
      g(θ_j) = L(y_k|θ_j) · f_{k-1}(θ_j)

   c. Normalize using trapezoidal rule:
      f_k(θ_j) = g(θ_j) / ∫ g(θ) dθ
      where ∫ g(θ) dθ ≈ Σ_m g(θ_m)Δθ

   d. Store posterior mean and MAP estimates
""")

def simulate_irt(n=20, theta_true=0.75):
    np.random.seed(42)
    a_vals = np.random.uniform(0.5, 2.0, n)
    b_vals = np.random.normal(0, 1, n)

    theta_grid = np.linspace(-4, 4, 200)
    prior = np.exp(-0.5*theta_grid**2)/np.sqrt(2*np.pi)
    prior /= np.trapezoid(prior, theta_grid)

    posterior_mean_est = []
    map_est = []

    for k in range(n):
        p_true = 1/(1 + np.exp(-a_vals[k]*(theta_true - b_vals[k])))
        y_k = 1 if np.random.random() < p_true else 0

        likelihood = (1/(1 + np.exp(-a_vals[k]*(theta_grid - b_vals[k]))))**y_k * \
                     (1 - 1/(1 + np.exp(-a_vals[k]*(theta_grid - b_vals[k]))))**(1-y_k)

        posterior = prior * likelihood
        posterior /= np.trapezoid(posterior, theta_grid)

        mean_est = np.trapezoid(theta_grid * posterior, theta_grid)
        map_est_val = theta_grid[np.argmax(posterior)]

        posterior_mean_est.append(mean_est)
        map_est.append(map_est_val)

        prior = posterior

    return posterior_mean_est, map_est

post_mean, map_est = simulate_irt()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(range(21)), y=[0.75]*21, mode='lines', name='θ_true = 0.75', line=dict(dash='dash')))
fig2.add_trace(go.Scatter(x=list(range(1,21)), y=post_mean, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=list(range(1,21)), y=map_est, mode='lines+markers', name='MAP'))
fig2.update_layout(title='IRT: Posterior Mean and MAP vs True Ability', xaxis_title='Step k', yaxis_title='θ Estimate', height=500, width=800)
fig2.show()

print("""
Analysis:
As k increases, both estimators converge toward θ_true = 0.75.
The initial prior N(0,1) is quickly overwhelmed by observed responses.
After approximately 10 items, the platform has high confidence in the user's ability.
""")

Q1. Bayesian Estimation of a User Ability Parameter from Item Responses

1. Visualizing the Mechanics:

P(Y_i = 1 | Θ = θ) = p_i(θ) = 1 / (1 + e^{-a_i(θ - b_i)})

where:
- a_i > 0 is the discrimination parameter
- b_i is the difficulty parameter

Interpretation:
- Increasing b_i shifts the curve right (harder item)
- Increasing a_i makes the curve steeper (more discriminating)




2. Sequential Likelihood Contribution:

For a single response y_k:
L(y_k | θ) = [p_k(θ)]^{y_k} [1 - p_k(θ)]^{1-y_k}

where y_k ∈ {0, 1}

For the running history vector y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = ∏_{i=1}^k [p_i(θ)]^{y_i} [1 - p_i(θ)]^{1-y_i}

3. Mathematical Formulation of the Running Update:

f_{Θ | Y^(k)}(θ | y^(k)) ∝ L(y_k | θ) · f_{Θ | Y^(k-1)}(θ | y^(k-1))

This recursive relationship uses the posterior from step (k-1) as the prior for step k.

4. Dynamic Shifting:

When y_k = 1 (correct answer) to a difficult item (large b_k):
- p_k(θ) is small for low θ and large for high θ
- Multiplying the prior by p_k(θ) shifts probability mass rightward
- The posterior peak moves toward higher θ values

When y_k = 0 (incorrect answer):
- [1 - p_k(θ)] is large for low θ and small for high θ
- The posterior peak shifts leftward

5. Tracking Certainty and Sharpness:

The discrimination parameter a_k controls the likelihood function shape:

Case 1: Very large a_k (a_k → ∞)
- p_


Analysis:
As k increases, both estimators converge toward θ_true = 0.75.
The initial prior N(0,1) is quickly overwhelmed by observed responses.
After approximately 10 items, the platform has high confidence in the user's ability.



In [9]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
import plotly.io as pio
pio.renderers.default = 'colab'

print("Q2. Bayesian Tracking of Click-Through Rates (CTR)")
print("="*70)

print("""
1. Structural Probability and Properties:

Beta Distribution PDF:
f_Θ(θ) = [θ^{α-1} (1-θ)^{β-1}] / B(α, β)

where B(α, β) = Γ(α)Γ(β) / Γ(α+β) is the Beta function.

Properties:
- Mean: E[Θ] = α / (α + β)
- Mode: (α - 1) / (α + β - 2) for α, β > 1
- Variance: αβ / [(α + β)²(α + β + 1)]

Interpretation of parameter pairs:
- (α=1, β=1): Uniform (uninformative) - all θ equally likely
- (α=2, β=8): Right-skewed - higher probability of smaller θ
- (α=8, β=2): Left-skewed - higher probability of larger θ
""")

theta_range = np.linspace(0.001, 0.999, 500)

fig = go.Figure()
params = [(1,1,'Uninformative'), (2,8,'Right-skewed'), (8,2,'Left-skewed')]
for alpha, beta_val, label in params:
    pdf = beta.pdf(theta_range, alpha, beta_val)
    fig.add_trace(go.Scatter(x=theta_range, y=pdf, mode='lines', name=f'Beta({alpha},{beta_val})'))

fig.update_layout(title='Beta Distributions', xaxis_title='θ', yaxis_title='Density', height=400, width=800)
fig.show()

print("""
2. Sequential Likelihood and Joint History:

For a single response y_k:
L(y_k | θ) = θ^{y_k} (1-θ)^{1-y_k}

For the running history y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = θ^{Σ_{i=1}^k y_i} (1-θ)^{k - Σ_{i=1}^k y_i}

3. Closed-Form Analytical Updates (Conjugacy):

Using Bayes' Theorem:
f(θ | y^(k)) ∝ L(y^(k) | θ) · f(θ)

= θ^{Σ y_i} (1-θ)^{k-Σ y_i} · θ^{α₀-1} (1-θ)^{β₀-1}

= θ^{α₀ + Σ y_i - 1} (1-θ)^{β₀ + k - Σ y_i - 1}

Therefore, the posterior is Beta(α_k, β_k) with:

α_k = α_{k-1} + y_k
β_k = β_{k-1} + (1 - y_k)

Posterior Mean:
E[Θ | Y^(k) = y^(k)] = α_k / (α_k + β_k)

4. Dynamic Shifting Mechanics:

When y_k = 1 (click observed):
- α_k = α_{k-1} + 1
- The mean increases: E[θ] = α_k/(α_k+β_k) > α_{k-1}/(α_{k-1}+β_{k-1})
- The distribution shifts right

When y_k = 0 (no click observed):
- β_k = β_{k-1} + 1
- The mean decreases
- The distribution shifts left

Comparison to non-conjugate setups:
- In conjugate models, updates are exact and closed-form
- In non-conjugate models (like 2PL IRT), numerical integration is required
- Computational cost: conjugate = O(1), non-conjugate = O(M) per step

5. Running Point Estimators:

Posterior Mean (Bayes estimate under squared error loss):
θ̂_Bayes^(k) = E[Θ | Y^(k)] = α_k / (α_k + β_k)

Maximum A Posteriori (MAP) estimate:
θ̂_MAP^(k) = argmax_θ f(θ | y^(k)) = (α_k - 1) / (α_k + β_k - 2)
provided α_k > 1 and β_k > 1
""")

def simulate_ctr(n=100, theta_true=0.35, alpha0=1, beta0=1):
    np.random.seed(42)
    alpha_k, beta_k = alpha0, beta0
    mean_est = []
    map_est = []

    for k in range(1, n+1):
        y_k = 1 if np.random.random() < theta_true else 0
        alpha_k += y_k
        beta_k += (1 - y_k)

        mean_est.append(alpha_k/(alpha_k+beta_k))
        if alpha_k > 1 and beta_k > 1:
            map_est.append((alpha_k-1)/(alpha_k+beta_k-2))
        else:
            map_est.append(alpha_k/(alpha_k+beta_k))

    return mean_est, map_est

mean_est, map_est = simulate_ctr()

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(101)), y=[0.35]*101, mode='lines', name='θ_true = 0.35', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=list(range(1,101)), y=mean_est, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=list(range(1,101)), y=map_est, mode='lines', name='MAP'))
fig.update_layout(title='CTR: Posterior Mean and MAP vs True CTR', xaxis_title='Impression k', yaxis_title='θ Estimate', height=500, width=800)
fig.show()

print("""
Analysis:
Both estimators converge to θ_true = 0.35 as k increases.
Initial uniform prior (α₀ = β₀ = 1) is quickly overwhelmed by data.
After approximately 50 impressions, estimates stabilize near true value.
Narrowing posterior variance indicates increased confidence.
""")

Q2. Bayesian Tracking of Click-Through Rates (CTR)

1. Structural Probability and Properties:

Beta Distribution PDF:
f_Θ(θ) = [θ^{α-1} (1-θ)^{β-1}] / B(α, β)

where B(α, β) = Γ(α)Γ(β) / Γ(α+β) is the Beta function.

Properties:
- Mean: E[Θ] = α / (α + β)
- Mode: (α - 1) / (α + β - 2) for α, β > 1
- Variance: αβ / [(α + β)²(α + β + 1)]

Interpretation of parameter pairs:
- (α=1, β=1): Uniform (uninformative) - all θ equally likely
- (α=2, β=8): Right-skewed - higher probability of smaller θ
- (α=8, β=2): Left-skewed - higher probability of larger θ




2. Sequential Likelihood and Joint History:

For a single response y_k:
L(y_k | θ) = θ^{y_k} (1-θ)^{1-y_k}

For the running history y^(k) = (y_1, y_2, ..., y_k):
L(y^(k) | θ) = θ^{Σ_{i=1}^k y_i} (1-θ)^{k - Σ_{i=1}^k y_i}

3. Closed-Form Analytical Updates (Conjugacy):

Using Bayes' Theorem:
f(θ | y^(k)) ∝ L(y^(k) | θ) · f(θ)

= θ^{Σ y_i} (1-θ)^{k-Σ y_i} · θ^{α₀-1} (1-θ)^{β₀-1}

= θ^{α₀ + Σ y_i - 1} (1-θ)^{β₀ + k - Σ y_i - 1}

Therefore, the posterior is Beta(α_k, β_k) with:

α_k = α_{k-1} + y_k
β_k = β_{k-1} + (1 - y_k)

Posterior Mean:
E[Θ | Y^(k) = y^(k)] = α_k / (α_k + β_k)

4. Dynamic Shifting Mechanics:

When y_k = 1 (click observed):
- α_k = α_{k-1} + 1
- The mean increases: E[θ] = α_k/(α_k+β_k) > α_{k-1}/(α_{k-1}+β_{k-1})
- The distribution shifts right

When y_k = 0 (no click observed):
- β_k = β_{k-1} + 1
- The mean decreases
- The distribution shifts left

Comparison to non-conjugate setups:
- In conjugate models, updates are exact and closed-form
- In non-conjugate models (


Analysis:
Both estimators converge to θ_true = 0.35 as k increases.
Initial uniform prior (α₀ = β₀ = 1) is quickly overwhelmed by data.
After approximately 50 impressions, estimates stabilize near true value.
Narrowing posterior variance indicates increased confidence.



In [8]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import beta, norm
import plotly.io as pio
pio.renderers.default = 'colab'

print("Q3. Bayesian Estimations for Structural Health Monitoring")
print("="*70)

print("""
1. Prior Belief Boundaries:

Θ ~ Beta(8, 1.5) on θ ∈ (0, 1]

Expected Prior Stiffness:
E[Θ] = α / (α + β) = 8 / (8 + 1.5) = 8 / 9.5 = 0.8421

Why this prior is appropriate:
- Mean ≈ 0.84 indicates high initial belief of structural health
- Small variance (from α+β = 9.5) shows confidence in manufacturing quality
- Bounded support [0, 1] matches physical constraint of stiffness efficiency
- β < α creates left-skewed distribution (slightly more mass near 1.0)
""")

theta_grid = np.linspace(0.01, 1.0, 500)
prior = beta.pdf(theta_grid, 8, 1.5)
prior /= np.trapezoid(prior, theta_grid)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior, mode='lines', name='Beta(8,1.5) Prior'))
fig.add_vline(x=8/9.5, line_dash="dash", line_color="red", annotation_text="E[θ] = 0.842")
fig.update_layout(title='SHM Prior Distribution', xaxis_title='θ (Stiffness Efficiency)', yaxis_title='Density', height=400, width=800)
fig.show()

print(f"E[θ] = {8/(8+1.5):.4f}")

print("""
2. Structural Likelihood Formulation:

Measurement Model:
y_k = θ · K_nominal · e^{ε_k}, where ε_k ~ N(0, σ²)

Taking the natural logarithm:
ln(y_k) = ln(θ) + ln(K_nominal) + ε_k

Therefore:
ln(y_k) | θ ~ N(ln(θ) + ln(K_nominal), σ²)

Likelihood function:
L(y_k | θ) = 1 / (y_k · σ · √(2π)) · exp[- (ln(y_k / (θ · K_nominal)))² / (2σ²)]

Joint Likelihood for running history:
L(y^(k) | θ) = ∏_{i=1}^k L(y_i | θ)

3. Mathematical Formulation of Non-Conjugate Grid Update:

Why no closed-form solution exists:
- Prior is Beta: f(θ) ∝ θ^{α-1} (1-θ)^{β-1}
- Likelihood is Log-normal: L(y|θ) ∝ (1/θ) · exp[- (ln(y/(θ·K_nominal)))²/(2σ²)]
- Product: Beta × Log-normal does not simplify to any standard distribution family

Recursive posterior update:
f(θ | y^(k)) ∝ L(y_k | θ) · f(θ | y^(k-1))

4. Running Point Estimates:

Posterior Mean (Bayes estimate):
θ̂_Bayes^(k) = E[Θ | y^(k)] = ∫₀¹ θ · f(θ | y^(k)) dθ

Maximum A Posteriori (MAP) estimate:
θ̂_MAP^(k) = argmax_{θ ∈ (0,1]} f(θ | y^(k))

5. Algorithmic Grid Approximation:

Step-by-step numerical procedure:

1. Define a discrete grid over θ ∈ (0, 1]:
   Θ_grid = {θ_1, θ_2, ..., θ_M} with uniform spacing Δθ

2. Initialize prior density on the grid:
   p_0(θ_j) = Beta(θ_j | 8, 1.5)

3. Normalize the prior:
   p_0(θ_j) = p_0(θ_j) / Σ_m p_0(θ_m) · Δθ

4. For each new sensor reading y_k (k = 1, 2, ..., n):

   a. Compute likelihood at each grid point:
      L(y_k | θ_j) = 1/(y_k·σ·√(2π)) · exp[- (ln(y_k/(θ_j·K_nominal)))²/(2σ²)]
      (Set L = 0 for θ_j where the expression is invalid)

   b. Compute unnormalized posterior:
      g(θ_j) = L(y_k | θ_j) · p_{k-1}(θ_j)

   c. Normalize using trapezoidal rule:
      Z_k = Σ_j g(θ_j) · Δθ
      p_k(θ_j) = g(θ_j) / Z_k

   d. Handle boundaries:
      - θ = 0 is excluded (log undefined)
      - θ = 1 is included as the healthy state
      - A small epsilon (e.g., 0.001) can be used for numerical stability

   e. Compute and store point estimates:
      θ̂_Bayes^(k) = Σ_j θ_j · p_k(θ_j) · Δθ
      θ̂_MAP^(k) = θ_j* where p_k(θ_j*) is maximal
""")

def simulate_shm(n=15, theta_true=0.68, K_nominal=50.0, sigma=0.15):
    np.random.seed(42)

    theta_grid = np.linspace(0.01, 1.0, 300)
    prior = beta.pdf(theta_grid, 8, 1.5)
    prior /= np.trapezoid(prior, theta_grid)

    mean_est = []
    map_est = []
    posterior_curves = []

    posterior_curves.append(prior)

    for k in range(n):
        eps = np.random.normal(0, sigma)
        y_k = theta_true * K_nominal * np.exp(eps)

        log_ratio = np.log(y_k / (theta_grid * K_nominal))
        likelihood = (1/(y_k * sigma * np.sqrt(2*np.pi))) * np.exp(-log_ratio**2/(2*sigma**2))
        likelihood[np.isnan(likelihood)] = 0

        posterior = prior * likelihood
        posterior /= np.trapezoid(posterior, theta_grid)

        mean_est.append(np.trapezoid(theta_grid * posterior, theta_grid))
        map_est.append(theta_grid[np.argmax(posterior)])

        if k in [0, 1, 2, 5, 10, 14]:
            posterior_curves.append(posterior)

        prior = posterior

    return mean_est, map_est, posterior_curves

mean_est, map_est, posterior_curves = simulate_shm()

fig = go.Figure()
milestones = [0, 1, 2, 5, 10, 15]
colors = ['blue', 'cyan', 'green', 'yellow', 'orange', 'red']
for i, (milestone, curve) in enumerate(zip(milestones, posterior_curves)):
    fig.add_trace(go.Scatter(x=theta_grid, y=curve, mode='lines', name=f'k={milestone}', line=dict(color=colors[i])))

fig.add_vline(x=0.68, line_dash="dash", line_color="black", annotation_text="θ_true = 0.68")
fig.update_layout(title='SHM: Posterior Density Evolution', xaxis_title='θ (Stiffness Efficiency)', yaxis_title='Density', height=500, width=800)
fig.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=list(range(16)), y=[0.68]*16, mode='lines', name='θ_true = 0.68', line=dict(dash='dash')))
fig2.add_trace(go.Scatter(x=list(range(1,16)), y=mean_est, mode='lines+markers', name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=list(range(1,16)), y=map_est, mode='lines+markers', name='MAP'))
fig2.update_layout(title='SHM: Convergence of Estimators', xaxis_title='Inspection Step k', yaxis_title='θ Estimate', height=500, width=800)
fig2.show()

print("""
Analysis:
- The initial optimistic prior (E[θ] = 0.842) is overcome after approximately 5-7 readings
- The posterior distribution narrows significantly after approximately 10 readings
- MAP converges faster than Mean (MAP mode moves more quickly to θ_true)
- Narrowing density implies engineers can confidently set safety thresholds
- At k = 15, the distribution is tightly concentrated near θ = 0.68
""")

Q3. Bayesian Estimations for Structural Health Monitoring

1. Prior Belief Boundaries:

Θ ~ Beta(8, 1.5) on θ ∈ (0, 1]

Expected Prior Stiffness:
E[Θ] = α / (α + β) = 8 / (8 + 1.5) = 8 / 9.5 = 0.8421

Why this prior is appropriate:
- Mean ≈ 0.84 indicates high initial belief of structural health
- Small variance (from α+β = 9.5) shows confidence in manufacturing quality
- Bounded support [0, 1] matches physical constraint of stiffness efficiency
- β < α creates left-skewed distribution (slightly more mass near 1.0)



E[θ] = 0.8421

2. Structural Likelihood Formulation:

Measurement Model:
y_k = θ · K_nominal · e^{ε_k}, where ε_k ~ N(0, σ²)

Taking the natural logarithm:
ln(y_k) = ln(θ) + ln(K_nominal) + ε_k

Therefore:
ln(y_k) | θ ~ N(ln(θ) + ln(K_nominal), σ²)

Likelihood function:
L(y_k | θ) = 1 / (y_k · σ · √(2π)) · exp[- (ln(y_k / (θ · K_nominal)))² / (2σ²)]

Joint Likelihood for running history:
L(y^(k) | θ) = ∏_{i=1}^k L(y_i | θ)

3. Mathematical Formulation of Non-Conjugate Grid Update:

Why no closed-form solution exists:
- Prior is Beta: f(θ) ∝ θ^{α-1} (1-θ)^{β-1}
- Likelihood is Log-normal: L(y|θ) ∝ (1/θ) · exp[- (ln(y/(θ·K_nominal)))²/(2σ²)]
- Product: Beta × Log-normal does not simplify to any standard distribution family

Recursive posterior update:
f(θ | y^(k)) ∝ L(y_k | θ) · f(θ | y^(k-1))

4. Running Point Estimates:

Posterior Mean (Bayes estimate):
θ̂_Bayes^(k) = E[Θ | y^(k)] = ∫₀¹ θ · f(θ | y^(k)) dθ

Maximum A Posteriori (MAP) estimate:
θ̂_MAP^(k) = argmax_{θ ∈ (0,1]} f(θ | y^(k


Analysis:
- The initial optimistic prior (E[θ] = 0.842) is overcome after approximately 5-7 readings
- The posterior distribution narrows significantly after approximately 10 readings
- MAP converges faster than Mean (MAP mode moves more quickly to θ_true)
- Narrowing density implies engineers can confidently set safety thresholds
- At k = 15, the distribution is tightly concentrated near θ = 0.68



In [7]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.io as pio
pio.renderers.default = 'colab'

print("Q4. Gaussian Mixture Clustering as Conditional Updating")
print("="*70)

print("""
1. Deriving the Marginal Density:

Using the law of total probability:
p(x_i) = Σ_{k=1}^K P(C_i = k) · p(x_i | C_i = k)

Substituting the Gaussian model:
p(x_i) = Σ_{k=1}^K φ_k · N(x_i | μ_k, Σ_k)

where:
- φ_k = P(C_i = k) is the prior probability of cluster k
- N(x_i | μ_k, Σ_k) is the multivariate Gaussian density

This is called a Gaussian mixture density because it is a convex combination
(weighted sum) of K Gaussian component densities.

2. Deriving the Posterior Cluster Probability:

Using Bayes' rule:
P(C_i = k | X_i = x_i) = P(X_i = x_i | C_i = k) · P(C_i = k) / P(X_i = x_i)

Substituting the Gaussian model and marginal density:
γ_{ik} = P(C_i = k | X_i = x_i)
       = [φ_k · N(x_i | μ_k, Σ_k)] / [Σ_{j=1}^K φ_j · N(x_i | μ_j, Σ_j)]

Interpretation: γ_{ik} is the "responsibility" of cluster k for data point i.

3. One-Hot Encoding of the Latent Cluster Variable:

Define Z_i = [Z_{i1}, Z_{i2}, ..., Z_{iK}]^T where:
Z_{ik} = 1 if C_i = k, and 0 otherwise

Conditional expectation:
E[Z_{ik} | X_i = x_i] = 1 · P(C_i = k | X_i = x_i) + 0 · P(C_i ≠ k | X_i = x_i)
                      = P(C_i = k | X_i = x_i) = γ_{ik}

Therefore:
E[Z_i | X_i = x_i] = [γ_{i1}, γ_{i2}, ..., γ_{iK}]^T

Conclusion: Soft cluster assignment is precisely the conditional expectation
of the one-hot encoded latent variable.

4. From Soft Assignment to Hard Clustering:

Soft assignment: E[Z_i | X_i = x_i] gives probabilities for each cluster

Hard assignment: Choose the most probable cluster:
Ĉ_i = argmax_{1 ≤ k ≤ K} γ_{ik}

Difference:
- Soft clustering: Each point has fractional membership in all clusters
- Hard clustering: Each point belongs to exactly one cluster

5. Conditional Expectation of the Observation Given the Cluster:

E[X_i | C_i = k] = ∫ x_i · p(x_i | C_i = k) dx_i = μ_k

Interpretation: μ_k is the center (mean) of cluster k.

Comparison of conditional expectations:

E[Z_i | X_i = x_i] → Soft membership of an observed point (posterior)
E[X_i | C_i = k] → Mean location of a cluster (parameter)

The first is a function of the data, the second is a model parameter.

6. The Complete-Data Likelihood:

Complete-data likelihood (if z_{ik} were known):
p(x_1, ..., x_n, z_1, ..., z_n) = ∏_{i=1}^n ∏_{k=1}^K [φ_k · N(x_i | μ_k, Σ_k)]^{z_{ik}}

Complete-data log-likelihood:
ℓ_c = Σ_{i=1}^n Σ_{k=1}^K z_{ik} [log φ_k + log N(x_i | μ_k, Σ_k)]

If z_{ik} were known, this would be easy to maximize because:
- The log φ_k terms separate by k
- The Gaussian log-likelihood terms separate by k
- Each cluster's parameters can be estimated independently

7. The EM Interpretation:

E-step: Replace unknown z_{ik} by their conditional expectations:
z_{ik} → E[Z_{ik} | X_i = x_i] = γ_{ik}

Expected complete-data log-likelihood:
Q = Σ_{i=1}^n Σ_{k=1}^K γ_{ik} [log φ_k + log N(x_i | μ_k, Σ_k)]

Interpretation of E-step:
The E-step is a conditional update of cluster membership probabilities.
It computes the posterior probability γ_{ik} given current parameter estimates.

8. Parameter Updates:

Maximizing Q with respect to parameters gives:

N_k = Σ_{i=1}^n γ_{ik}  (effective number of points in cluster k)

φ_k^{new} = N_k / n  (mixing weight update)

μ_k^{new} = (1/N_k) Σ_{i=1}^n γ_{ik} x_i  (mean update)

Σ_k^{new} = (1/N_k) Σ_{i=1}^n γ_{ik} (x_i - μ_k^{new})(x_i - μ_k^{new})^T

Interpretation of γ_{ik}:
γ_{ik} acts as a fractional membership weight of observation x_i in cluster k.
Each point contributes to all clusters, weighted by its responsibility.

9. Interpretation:

GMM clustering can be viewed as a repeated process of conditional updating:

- The mixture weight φ_k is the prior probability of cluster k
- The Gaussian density N(x_i | μ_k, Σ_k) measures how compatible x_i is with cluster k
- The responsibility γ_{ik} is the posterior probability of cluster k after observing x_i
- The soft assignment vector is E[Z_i | X_i = x_i]
- The M-step updates parameters using posterior membership probabilities as weights

Conclusion: Gaussian mixture clustering is probabilistic clustering based on
conditional expectations of latent cluster membership variables.
""")

class GMMFinancialSegmenter:
    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.gmm = None
        self.scaler = None
        self.X_train = None
        self.X_test = None

    def load_and_split(self, X, test_size=0.2):
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X)
        self.X_train, self.X_test = train_test_split(X_scaled, test_size=test_size, random_state=self.random_state)
        return self.X_train, self.X_test

    def fit(self):
        self.gmm = GaussianMixture(n_components=self.n_components, random_state=self.random_state, max_iter=200)
        self.gmm.fit(self.X_train)
        print(f"Converged: {self.gmm.converged_}")
        print(f"Iterations: {self.gmm.n_iter_}")
        print(f"AIC: {self.gmm.aic(self.X_train):.2f}")
        print(f"BIC: {self.gmm.bic(self.X_train):.2f}")
        return self.gmm

    def test_log_likelihood(self):
        if self.gmm is None:
            raise ValueError("Model not fitted yet")
        return self.gmm.score(self.X_test)

    def create_heatmap(self):
        fig = make_subplots(rows=1, cols=2, subplot_titles=("2D Density Heatmap", "Marginal Distributions"))

        fig.add_trace(go.Histogram2dContour(x=self.X_train[:,0], y=self.X_train[:,1], colorscale='Viridis',
                                             contours=dict(coloring='heatmap')), row=1, col=1)

        fig.add_trace(go.Histogram(x=self.X_train[:,0], histnorm='probability density', name='Feature 1'), row=1, col=2)
        fig.add_trace(go.Histogram(y=self.X_train[:,1], histnorm='probability density', name='Feature 2'), row=1, col=2)

        fig.update_layout(height=500, width=1000, title_text="Training Data Visualization")
        return fig

    def plot_assignments(self, data, title):
        x_min, x_max = data[:,0].min() - 0.5, data[:,0].max() + 0.5
        y_min, y_max = data[:,1].min() - 0.5, data[:,1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
        grid = np.c_[xx.ravel(), yy.ravel()]

        probs = self.gmm.predict_proba(grid)
        max_resp = np.argmax(probs, axis=1).reshape(xx.shape)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=np.linspace(x_min, x_max, 100), y=np.linspace(y_min, y_max, 100),
                                  z=max_resp, colorscale='Viridis', showscale=True,
                                  name='Cluster Boundaries'))

        colors = ['red', 'blue', 'green']
        for k in range(self.n_components):
            idx = self.gmm.predict(data) == k
            fig.add_trace(go.Scatter(x=data[idx,0], y=data[idx,1], mode='markers',
                                      marker=dict(color=colors[k], size=6), name=f'Cluster {k+1}'))

        fig.update_layout(title=title, xaxis_title='Feature 1', yaxis_title='Feature 2', height=500, width=800)
        return fig

    def show_plots(self):
        heatmap_fig = self.create_heatmap()
        heatmap_fig.show()

        train_fig = self.plot_assignments(self.X_train, "Training Data: Soft Assignment Boundaries")
        train_fig.show()

        test_fig = self.plot_assignments(self.X_test, "Test Data: Soft Assignment Boundaries")
        test_fig.show()

np.random.seed(42)
n_samples = 500
X = np.vstack([
    np.random.multivariate_normal([-2, -2], [[0.5, 0.2], [0.2, 0.5]], int(n_samples*0.4)),
    np.random.multivariate_normal([0, 2], [[0.6, -0.2], [-0.2, 0.6]], int(n_samples*0.3)),
    np.random.multivariate_normal([3, 0], [[0.4, 0.3], [0.3, 0.4]], int(n_samples*0.3))
])

segmenter = GMMFinancialSegmenter(n_components=3)
segmenter.load_and_split(X)
segmenter.fit()

print(f"Test Log-Likelihood: {segmenter.test_log_likelihood():.2f}")
segmenter.show_plots()

print("""
Evaluation:
The contour map shows soft assignment regions where boundaries
represent areas of cluster ambiguity (γ ≈ 0.5 for multiple clusters).

The heatmap reveals the multimodal structure of the data.
Test data points follow the same cluster boundaries as training data,
validating the learned density functions generalize well.

Soft assignment expectation E[Z_i | X_i = x_grid] is visualized by
the continuous color gradients in the contour map, showing
probabilistic memberships rather than hard cutoffs.
""")

Q4. Gaussian Mixture Clustering as Conditional Updating

1. Deriving the Marginal Density:

Using the law of total probability:
p(x_i) = Σ_{k=1}^K P(C_i = k) · p(x_i | C_i = k)

Substituting the Gaussian model:
p(x_i) = Σ_{k=1}^K φ_k · N(x_i | μ_k, Σ_k)

where:
- φ_k = P(C_i = k) is the prior probability of cluster k
- N(x_i | μ_k, Σ_k) is the multivariate Gaussian density

This is called a Gaussian mixture density because it is a convex combination
(weighted sum) of K Gaussian component densities.

2. Deriving the Posterior Cluster Probability:

Using Bayes' rule:
P(C_i = k | X_i = x_i) = P(X_i = x_i | C_i = k) · P(C_i = k) / P(X_i = x_i)

Substituting the Gaussian model and marginal density:
γ_{ik} = P(C_i = k | X_i = x_i)
       = [φ_k · N(x_i | μ_k, Σ_k)] / [Σ_{j=1}^K φ_j · N(x_i | μ_j, Σ_j)]

Interpretation: γ_{ik} is the "responsibility" of cluster k for data point i.

3. One-Hot Encoding of the Latent Cluster Variable:

Define Z_i = [Z_{i1}, Z_{i2}, ..., Z_{iK}]^T where:
Z_{ik} 


Evaluation:
The contour map shows soft assignment regions where boundaries
represent areas of cluster ambiguity (γ ≈ 0.5 for multiple clusters).

The heatmap reveals the multimodal structure of the data.
Test data points follow the same cluster boundaries as training data,
validating the learned density functions generalize well.

Soft assignment expectation E[Z_i | X_i = x_grid] is visualized by
the continuous color gradients in the contour map, showing
probabilistic memberships rather than hard cutoffs.

